# Extract from DynamoDB

In [ ]:

# ── Cell 1: Imports ───────────────────────────────────────────────────────────
import boto3
import json
from dotenv import load_dotenv
import os, gzip

REGION   = "ap-southeast-2"
MODEL_ID = "anthropic.claude-3-5-sonnet-20241022-v2:0"


load_dotenv(dotenv_path=r"/home/sagemaker-user/swtest1/llm_poc/.env", override=True)
DEV_ROLE = os.getenv("DEV_ROLE")  
print(f"DEV_ROLE: {DEV_ROLE}") 


In [ ]:
sts_client   = boto3.client("sts")
sts_session  = sts_client.assume_role(
    RoleArn=DEV_ROLE,
    RoleSessionName="notebook-session"
)
creds = sts_session["Credentials"]

dynamodb_client = boto3.client(
    "dynamodb",
    region_name=REGION,
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
)

def lookup_merchant(merchant_name: str) -> dict:
    normalised = merchant_name.upper().strip()
    escaped = normalised.translate(str.maketrans({"'": r"''"}))
    try:
        resp = dynamodb_client.execute_statement(
            Statement=(
                "SELECT * FROM mapping "
                "WHERE \"type\" = 'merchant' "
                f"AND \"entity\" = '{escaped}'"
            )
        )
        items = resp.get("Items", [])
        if items:
            item = items[0]
            def unwrap(v):
                # DynamoDB typed values: {"S": "foo"} → "foo"; missing fields → None
                if isinstance(v, dict) and v:
                    return next(iter(v.values()))
                return None
            return {
                "found":    True,
                "entity":   unwrap(item.get("entity")),
                "category": unwrap(item.get("category")),
                "mapping":  unwrap(item.get("mapping")),
                "priority": unwrap(item.get("priority")),
            }
        return {"found": False, "entity": normalised}
    except Exception as e:
        return {"found": False, "error": str(e)}

# Quick test — swap "COLES" for any entity you know exists in your table
result = lookup_merchant("COLES")
print(result)
# Expected if access is fine: {"found": True, "entity": "COLES", "category": "GROCERIES", ...}
# If you see {"found": False, "error": "..."} — check IAM permissions



In [ ]:
result = lookup_merchant("WOOLWORTHS")  # ← swap for a known entity in your table
print(result)

In [ ]:
resp = dynamodb_client.execute_statement(
    Statement="SELECT * FROM mapping WHERE \"entity\" = 'COLES'"
)
print(resp.get("Items", []))

In [ ]:
# What identity are we using after role assumption?
sts = boto3.client("sts")
print(sts.get_caller_identity())

In [ ]:
# Use the same creds from the role assumption
sts_assumed = boto3.client(
    "sts",
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
)
print(sts_assumed.get_caller_identity())

In [ ]:
# What role did we assume?
print(DEV_ROLE)

# And confirm the assumption succeeded — what account is it in?
print(sts_session["AssumedRoleUser"]["Arn"])

In [ ]:
# Scan a small sample to see what entities are in this environment
resp = dynamodb_client.execute_statement(
    Statement="SELECT * FROM mapping WHERE \"type\" = 'merchant'"
)
for item in resp.get("Items", [])[:10]:
    print({k: list(v.values())[0] for k, v in item.items()})

In [ ]:
resp = dynamodb_client.describe_table(TableName="mapping")
print("Approx item count:", resp["Table"]["ItemCount"])
print("Table ARN:", resp["Table"]["TableArn"])

In [ ]:
resp = dynamodb_client.execute_statement(
    Statement="SELECT * FROM mapping WHERE \"type\" = 'merchant' AND begins_with(\"entity\", 'APPLE')"
)
for item in resp.get("Items", []):
    print({k: list(v.values())[0] for k, v in item.items()})

In [ ]:
print(lookup_merchant("APPLE"))
print(lookup_merchant("COLES"))
print(lookup_merchant("JB"))  # should return found: False

In [ ]:
resp = dynamodb_client.execute_statement(
    Statement="SELECT * FROM mapping WHERE \"type\" = 'merchant' AND \"entity\" = 'APPLE'"
)
print(resp)

In [ ]:
test_merchants = ["COLES", "WOOLWORTHS", "NETFLIX", "UBER", "BUNNINGS", "MEDICARE", "ATO"]
for m in test_merchants:
    result = lookup_merchant(m)
    print(f"{m}: {result}")